### Imports

In [1]:
#!pip install ucimlrepo
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

### Data Acquisition

In [2]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
communities_and_crime = fetch_ucirepo(id=183) 
  
# data (as pandas dataframes) 
X = pd.DataFrame(communities_and_crime.data.features)
y = pd.DataFrame(communities_and_crime.data.targets)
  
# metadata 
#print(communities_and_crime.metadata) 
  
# variable information 
#print(communities_and_crime.variables) 

df = pd.concat([X, y], axis=1)

### Preprocessing

In [3]:
def preprocess_communities_crime(X, y, missing_col_thresh=0.5, drop_cols=True):
    # copying so original data is untouched
    X = X.copy()
    y = y.copy()

    # making y a Series because it is a single column dataframe
    if isinstance(y, pd.DataFrame):
        y = y.squeeze()

    # combining so rows stay aligned if target rows are dropped
    df = pd.concat([X, y], axis=1)

    # drop rows with missing target
    df = df.dropna(subset=['ViolentCrimesPerPop'])

    # separating back
    y_clean = df['ViolentCrimesPerPop']
    X_clean = df.drop(columns=['ViolentCrimesPerPop'])

    # converting object columns to numeric, coercing "?" etc. to NaN
    obj_cols = X_clean.select_dtypes(include='object').columns
    for col in obj_cols:
        X_clean[col] = pd.to_numeric(X_clean[col], errors='coerce')

    # only dropping ID / non-predictive columns if approved
    cols_to_drop = []
    candidate_cols = ['communityname', 'county', 'community', 'state', 'fold']
    if drop_cols:
        cols_to_drop = [col for col in candidate_cols if col in X_clean.columns]

    # dropping columns with too much missingness
    missing_pct = X_clean.isna().mean()
    high_missing_cols = missing_pct[missing_pct > missing_col_thresh].index.tolist()

    # combine and remove duplicates
    dropped_cols = list(dict.fromkeys(cols_to_drop + high_missing_cols))

    # actually drop all chosen columns
    X_clean = X_clean.drop(columns=dropped_cols, errors='ignore')

    # recomputing missingness after dropping columns
    missing_summary = X_clean.isna().mean().sort_values(ascending=False)

    return X_clean, y_clean, dropped_cols, missing_summary

X_clean, y_clean, dropped_cols, missing_summary = preprocess_communities_crime(X, y)
X_clean = X_clean.dropna()
y_clean = y_clean.loc[X_clean.index]

print("X shape:", X_clean.shape)
print("y shape:", y_clean.shape)
print("\nDropped columns:")
print(dropped_cols)
print("\nTop remaining missingness:")
print(missing_summary.head(20))

X shape: (1993, 100)
y shape: (1993,)

Dropped columns:
['communityname', 'county', 'community', 'state', 'fold', 'LemasSwornFT', 'LemasSwFTPerPop', 'LemasSwFTFieldOps', 'LemasSwFTFieldPerPop', 'LemasTotalReq', 'LemasTotReqPerPop', 'PolicReqPerOffic', 'PolicPerPop', 'RacialMatchCommPol', 'PctPolicWhite', 'PctPolicBlack', 'PctPolicHisp', 'PctPolicAsian', 'PctPolicMinor', 'OfficAssgnDrugUnits', 'NumKindsDrugsSeiz', 'PolicAveOTWorked', 'PolicCars', 'PolicOperBudg', 'LemasPctPolicOnPatr', 'LemasGangUnitDeploy', 'PolicBudgPerPop']

Top remaining missingness:
OtherPerCap            0.000502
PersPerOccupHous       0.000000
PctVacantBoarded       0.000000
PctHousOwnOcc          0.000000
PctHousOccup           0.000000
HousVacant             0.000000
MedNumBR               0.000000
PctHousLess3BR         0.000000
PctPersDenseHous       0.000000
PctPersOwnOccup        0.000000
PersPerRentOccHous     0.000000
PersPerOwnOccHous      0.000000
PctLargHouseOccup      0.000000
NumImmig               0

### Post cleaning Audit

In [4]:
print(X_clean.shape)
print(y_clean.shape)
print(y_clean.describe())
print(X_clean.isna().sum().sum())   # total missing values left
print(X_clean.duplicated().sum())   # duplicate rows

(1993, 100)
(1993,)
count    1993.000000
mean        0.237983
std         0.233043
min         0.000000
25%         0.070000
50%         0.150000
75%         0.330000
max         1.000000
Name: ViolentCrimesPerPop, dtype: float64
0
0


### Train/Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y_clean, test_size=0.2, random_state=427
)

X_train.to_csv("Data/X_train")
X_test.to_csv("Data/X_test")
y_train.to_csv("Data/y_train")
y_test.to_csv("Data/y_test")